# Figure 1 - Panel F: biological-process enrichment ridgeplot (v2)

Ridgeplot of 11 curated ORA-significant GO biological-process terms. **Change vs v1:** the
density now uses **every gene in the gene set that has an LRT p-value** (not only the GSEA
core-enrichment genes) - so a gene set with no real shift shows a flat/left-piled density,
while a genuinely enriched set shifts right. x = per-gene -log10(LRT p); fill = the set's
**ORA p.adjust** on a single shared, log-scaled colour ramp; dashed line at -log10(0.05).

- Backs draft **Section 1.3 (Figure 1F)**. Kernel: `r_downstream`.
- Input: `ORA/{gsea,ora}_go_neglog10pval_chisq_optimized2.rds`.
- Output: `.../Fig1/fig1f_bp_enrichment/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
    REPO_ROOT <- normalizePath(getwd())
    while (!file.exists(file.path(REPO_ROOT, "environment", "downstr.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
        REPO_ROOT <- dirname(REPO_ROOT)
    }
    Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))

In [ ]:
source(paste0(REPO_ROOT, "/figures/style/paper_fig_style.R"))
suppressPackageStartupMessages({
    library(clusterProfiler)
    library(dplyr)
    library(stringr)
    library(ggridges)
    library(scales)
})
ORA_DIR <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized/ORA")
PANEL <- "fig1f_bp_enrichment"
FIG_W <- HALF_W_IN
FIG_H <- mm_to_in(58) # 85 x 58 mm (half-grid)

gsea_go <- readRDS(file.path(
    ORA_DIR,
    "gsea_go_neglog10pval_chisq_optimized2.rds"
))
ora_go <- readRDS(file.path(
    ORA_DIR,
    "ora_go_neglog10pval_chisq_optimized2.rds"
))

# all member genes (with an LRT p-value) per term, + the term's ORA p.adjust for colour
build_long <- function(gsea_obj, ora_obj, descs) {
    gl <- gsea_obj@geneList # named -log10(p) over ALL tested genes
    gs <- gsea_obj@geneSets # term id -> gene ids
    gres <- gsea_obj@result
    ores <- ora_obj@result
    out <- list()
    for (d in descs) {
        gid <- ores$ID[match(d, ores$Description)]
        if (is.na(gid) || is.null(gs[[gid]])) {
            message("skip: ", d)
            next
        }
        genes <- intersect(gs[[gid]], names(gl))
        padj <- ores$p.adjust[match(d, ores$Description)]
        out[[d]] <- data.frame(
            term = d,
            neglog10p = as.numeric(gl[genes]),
            ora_padj = padj,
            n = length(genes),
            stringsAsFactors = FALSE
        )
    }
    do.call(rbind, out)
}

In [ ]:
curated_bio <- c(
    # RNA processing (collapsed from 11 near-duplicate splicing/mRNA terms -> 2 reps)
    "regulation of RNA splicing",
    "mRNA processing",
    # cognition / learning bridge to neurodevelopmental disease
    "cognition",
    "learning or memory",
    # proteostasis + translation
    "ubiquitin-dependent protein catabolic process",
    "cytoplasmic translation",
    # mitochondrial (bridges to the Fig 3 mitochondrial convergence) + broad neuro
    "apoptotic mitochondrial changes",
    "nervous system process"
)

df <- build_long(gsea_go, ora_go, curated_bio)
# label = wrapped term + n; keep the curated order (top -> bottom)
lab <- setNames(
    sprintf(
        "%s  (n=%d)",
        str_wrap(curated_bio, 28),
        sapply(curated_bio, function(d) df$n[match(d, df$term)])
    ),
    curated_bio
)
df$label <- factor(lab[df$term], levels = rev(lab[curated_bio]))
cat("genes per term:\n")
print(tapply(df$neglog10p, df$term, length)[curated_bio])

In [ ]:
brks <- signif(
    exp(seq(log(min(df$ora_padj)), log(max(df$ora_padj)), length.out = 3)),
    2
)
p <- ggplot(df, aes(x = neglog10p, y = label, fill = ora_padj)) +
    geom_density_ridges(
        scale = 1.5,
        alpha = 0.92,
        color = "grey35",
        linewidth = 0.25,
        rel_min_height = 0.008,
        bandwidth = 0.5
    ) +
    scale_fill_gradientn(
        colours = rev(c("#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026")),
        trans = "log10",
        limits = range(df$ora_padj),
        breaks = brks,
        labels = formatC(brks, format = "g", digits = 1),
        name = expression(ORA * " " * italic(p)[adj]),
        guide = guide_colourbar(
            barwidth = 0.45,
            barheight = 3.2,
            ticks.colour = "grey30"
        )
    ) +
    geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        color = "black",
        linewidth = 0.4
    ) +
    labs(
        x = expression(
            "-" * "log"[10] * " " * italic(p) * "-value  (per gene in set)"
        ),
        y = NULL
    ) +
    coord_cartesian(xlim = c(0, 12)) +
    theme_paper +
    theme(
        axis.text.y = element_text(size = FONT_LEGEND, lineheight = 0.85),
        legend.position = "right",
        panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"),
        panel.grid.major.y = element_blank()
    )

# TODO 12: all terms come from one database (GO Biological Process) -> a single grey
# database banner rather than repeating the DB name on every term.
p <- p +
    annotate(
        "label",
        x = 11.7,
        y = length(levels(df$label)) - 0.15,
        label = "GO: Biological Process",
        hjust = 1,
        vjust = 1,
        fill = "grey88",
        color = "grey20",
        label.size = 0,
        size = FONT_ANNOT / .pt,
        label.padding = unit(0.12, "lines")
    )
save_and_show(p, PANEL, FIG_W, FIG_H)

### Official GO term definitions

Each category shown in this panel is a GO Biological Process term (source: Gene Ontology / AmiGO).

- **regulation of RNA splicing** (GO:0043484): Any process that modulates the frequency, rate or extent of RNA splicing, the process of removing sections of the primary RNA transcript to remove sequences not present in the mature form of the RNA and joining the remaining sections to form the mature form of the RNA.
- **mRNA processing** (GO:0006397): Any process involved in the conversion of a primary mRNA transcript into one or more mature mRNA(s) prior to translation into polypeptide.
- **cognition** (GO:0050890): The operation of the mind by which an organism becomes aware of objects of thought or perception; it includes the mental activities associated with thinking, learning, and memory.
- **learning or memory** (GO:0007611): The acquisition and processing of information and/or the storage and retrieval of this information over time.
- **ubiquitin-dependent protein catabolic process** (GO:0006511): The chemical reactions and pathways resulting in the breakdown of a protein or peptide by hydrolysis of its peptide bonds, initiated by the covalent attachment of a ubiquitin group, or multiple ubiquitin groups, to the protein.
- **cytoplasmic translation** (GO:0002181): The chemical reactions and pathways resulting in the formation of a protein in the cytoplasm. This is a ribosome-mediated process in which the information in messenger RNA (mRNA) is used to specify the sequence of amino acids in the protein.
- **apoptotic mitochondrial changes** (GO:0008637): The morphological and physiological alterations undergone by mitochondria during apoptosis.
- **nervous system process** (GO:0050877): An organ system process carried out by any of the organs or tissues of the nervous system.


## Variant: chisq∩permutation intersection foreground + left database strip

Same GO-BP ridgeplot recoloured from the **chisq ∩ permutation** intersection ORA (mouse; genes
significant under both the LRT and the permutation null). The database name now sits in a vertical
strip on the left of the term labels (like fig1g), replacing the top-right annotation. Two of the
original eight terms are dropped (NS under the stricter foreground): _ubiquitin-dependent protein
catabolic process_ (0.11) and _nervous system process_ (0.18). Saved `fig1f_bp_enrichment__chisqperm`.


In [ ]:
# --- fig1f on the chisq∩perm intersection foreground; database name as a LEFT strip ---
INT_DIR <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized/ORA/chisq_perm_intersection")
ora_go_int <- readRDS(file.path(INT_DIR, "ora_go_neglog10pval_chisqperm.rds"))
cur_go_int <- c(
    "regulation of RNA splicing",
    "mRNA processing",
    "cognition",
    "learning or memory",
    "cytoplasmic translation",
    "apoptotic mitochondrial changes"
)
df_i <- build_long(gsea_go, ora_go_int, cur_go_int) # gsea_go + build_long from cell 1
lab_i <- setNames(
    sprintf(
        "%s  (n=%d)",
        str_wrap(cur_go_int, 28),
        sapply(cur_go_int, function(d) df_i$n[match(d, df_i$term)])
    ),
    cur_go_int
)
df_i$label <- factor(lab_i[df_i$term], levels = rev(lab_i[cur_go_int]))
df_i$database <- factor("GO: Biological Process")
brks_i <- signif(
    exp(seq(log(min(df_i$ora_padj)), log(max(df_i$ora_padj)), length.out = 3)),
    2
)
p_i <- ggplot(df_i, aes(x = neglog10p, y = label, fill = ora_padj)) +
    geom_density_ridges(
        scale = 1.4,
        alpha = 0.92,
        color = "grey35",
        linewidth = 0.25,
        rel_min_height = 0.008,
        bandwidth = 0.5
    ) +
    scale_fill_gradientn(
        colours = rev(c("#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026")),
        trans = "log10",
        limits = range(df_i$ora_padj),
        breaks = brks_i,
        labels = formatC(brks_i, format = "g", digits = 1),
        name = expression(ORA * " " * italic(p)[adj]),
        guide = guide_colourbar(
            barwidth = 0.45,
            barheight = 3.2,
            ticks.colour = "grey30"
        )
    ) +
    geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        color = "black",
        linewidth = 0.4
    ) +
    labs(
        x = expression(
            "-" * "log"[10] * " " * italic(p) * "-value  (per gene in set)"
        ),
        y = NULL
    ) +
    facet_grid(database ~ ., switch = "y") +
    scale_x_continuous(
        expand = expansion(mult = c(0, 0.02)),
        breaks = seq(0, 12.5, 2.5)
    ) +
    scale_y_discrete(expand = expansion(mult = c(0.02, 0.25))) +
    coord_cartesian(xlim = c(0, 12.5), clip = "on") +
    theme_paper +
    theme(
        axis.text.y = element_text(size = FONT_LEGEND, lineheight = 0.85),
        legend.position = "right",
        panel.border = element_rect(linewidth = 0.3, fill = NA),
        panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"),
        panel.grid.major.y = element_blank(),
        strip.text.y.left = element_text(
            size = FONT_TICK - 0.5,
            face = "bold",
            angle = 90
        ),
        strip.background = element_rect(fill = "#F0F0F0", linewidth = 0.4),
        strip.placement = "outside"
    )
save_and_show(p_i, "fig1f_bp_enrichment__chisqperm", FIG_W, FIG_H)

### Variant: quarter-width (labels inside ridges, top-right; no y-axis labels)


In [ ]:
# --- fig1f QUARTER-WIDTH variant: geneset labels moved INSIDE the ridges (top-right, on a faint
# white plate for contrast over dark ridges), y-axis labels removed, panel narrowed to quarter
# width (40.5 mm) so it tiles next to a quarter-width fig1_neuron_global_w. Reuses df_i/lab_i/
# cur_go_int/brks_i from the chisqperm cell. ---
lab_df <- data.frame(
    label = factor(lab_i[cur_go_int], levels = rev(lab_i[cur_go_int])),
    text = sapply(cur_go_int, function(d) {
        str_wrap(paste0(d, " (n=", df_i$n[match(d, df_i$term)], ")"), 13)
    }),
    stringsAsFactors = FALSE
)
p_q <- ggplot(df_i, aes(x = neglog10p, y = label, fill = ora_padj)) +
    geom_density_ridges(
        scale = 1.4,
        alpha = 0.92,
        color = "grey35",
        linewidth = 0.25,
        rel_min_height = 0.008,
        bandwidth = 0.5
    ) +
    geom_label(
        data = lab_df,
        aes(x = 12.3, y = label, label = text),
        inherit.aes = FALSE,
        hjust = 1,
        vjust = 1,
        nudge_y = 1.28,
        size = (FONT_TICK - 1) / 2.845,
        lineheight = 0.78,
        color = "grey10",
        fill = alpha("white", 0.62),
        label.size = 0,
        label.padding = unit(0.45, "pt")
    ) +
    scale_fill_gradientn(
        colours = rev(c("#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026")),
        trans = "log10",
        limits = range(df_i$ora_padj),
        breaks = brks_i,
        labels = formatC(brks_i, format = "g", digits = 1),
        name = expression(ORA * " " * italic(p)[adj]),
        guide = guide_colourbar(
            barwidth = 0.4,
            barheight = 2.6,
            ticks.colour = "grey30"
        )
    ) +
    geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        color = "black",
        linewidth = 0.4
    ) +
    labs(
        x = expression("-" * "log"[10] * " " * italic(p) * "-value"),
        y = NULL
    ) +
    facet_grid(database ~ ., switch = "y") +
    scale_x_continuous(
        expand = expansion(mult = c(0, 0.02)),
        breaks = seq(0, 12.5, 5)
    ) +
    scale_y_discrete(expand = expansion(mult = c(0.02, 0.42))) +
    coord_cartesian(xlim = c(0, 12.5), clip = "on") +
    theme_paper +
    theme(
        axis.text.y = element_blank(),
        axis.ticks.y = element_blank(),
        legend.position = "right",
        panel.border = element_rect(linewidth = 0.3, fill = NA),
        panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"),
        panel.grid.major.y = element_blank(),
        strip.text.y.left = element_text(
            size = FONT_TICK - 0.5,
            face = "bold",
            angle = 90
        ),
        strip.background = element_rect(fill = "#F0F0F0", linewidth = 0.4),
        strip.placement = "outside"
    )
save_and_show(
    p_q,
    "fig1f_bp_enrichment__chisqperm_quarter",
    QUARTER_W_IN,
    FIG_H
)